# Practice #6. "Neural Networks for Time Series Forecasting"

Prepare a series for a neural network without leaking the test set,
then build an MLP, an LSTM and a bidirectional LSTM in PyTorch.

Fill in the cells tagged `graded`, keeping every name and signature exactly as
given — they are graded automatically on shapes, scaling discipline and forward
passes, never on a trained model's score.

**Needs `torch`**, which a default Anaconda install does not include:
`pip install -r requirements.txt`.

**Runtime:** about 40 seconds on a laptop CPU for all three networks (MLP 60
epochs, LSTM and BiLSTM 40 each). No GPU. Cut the epoch counts while iterating.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import MinMaxScaler

def find_data_dir():
    """The repo's data/ directory, wherever the kernel happens to start."""
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "data" / "airline-passengers.csv").exists():
            return folder / "data"
    raise FileNotFoundError("data/ not found - run this notebook inside the repo")


DATA_DIR = globals().get("DATA_DIR", find_data_dir())
RANDOM_SEED = 42


def set_seed(seed=RANDOM_SEED):
    """Seed every generator the notebook touches, so runs are comparable."""
    np.random.seed(seed)
    torch.manual_seed(seed)


set_seed()

## 0. Data

In [ ]:
def load_series(path, time_col, value_col):
    """Read a CSV into a float Series named "y" on a DatetimeIndex."""
    # TODO: same contract as Practices 1-5.
    raise NotImplementedError

In [ ]:
series = load_series(DATA_DIR / "daily-min-temperatures.csv", "Date", "Temp")
print(f"{len(series)} points, {series.index.min():%Y-%m-%d} to {series.index.max():%Y-%m-%d}")

plt.figure(figsize=(20, 5))
plt.plot(series)
plt.title("Melbourne daily minimum temperatures");

## 1. Data preparation

Four steps, in this order:

1. **Split** by time — train, validation, test.
2. **Fit** the scaler on the training split only.
3. **Transform** all three splits with that scaler.
4. **Sequence** inside each split.

Fitting the scaler before the split is the most common leak in neural-network
time-series code, and a quiet one: the score just comes out slightly optimistic
and the loss curve looks fine.

In [ ]:
def split_series(series, train_frac=0.7, val_frac=0.15):
    """Split chronologically into (train, validation, test).

    Before scaling and before sequencing — everything downstream needs it.
    """
    # TODO
    raise NotImplementedError


def fit_scaler(train):
    """MinMaxScaler fitted on the TRAINING values only.

    Fit it on the whole series and the network gets the test set's min and max —
    numbers it will not have in production.
    """
    # TODO: MinMaxScaler wants 2-D; reshape with .to_numpy().reshape(-1, 1)
    raise NotImplementedError


def scale(scaler, values):
    """Apply a fitted scaler to a 1-D array/Series; return a 1-D array."""
    # TODO
    raise NotImplementedError


def create_sequences(data, seq_length, pred_length=1):
    """Slice a 1-D array into (X, y) sequences.

    X[i] = data[i : i+seq_length]          shape (n_samples, seq_length)
    y[i] = data[i+seq_length : +pred_length]  shape (n_samples, pred_length)
    """
    # TODO
    raise NotImplementedError

In [ ]:
train_raw, val_raw, test_raw = split_series(series)
scaler = fit_scaler(train_raw)

print(f"train {len(train_raw)}, val {len(val_raw)}, test {len(test_raw)}")
print(f"scaler learned min={scaler.data_min_[0]:.2f}, max={scaler.data_max_[0]:.2f} "
      f"(from the TRAIN split only)")
print(f"full-series min={series.min():.2f}, max={series.max():.2f}")

SEQ_LENGTH = 30
X_train, y_train = create_sequences(scale(scaler, train_raw), SEQ_LENGTH)
X_val, y_val = create_sequences(scale(scaler, val_raw), SEQ_LENGTH)
X_test, y_test = create_sequences(scale(scaler, test_raw), SEQ_LENGTH)
print(f"\nX_train {X_train.shape}, X_val {X_val.shape}, X_test {X_test.shape}")

In [ ]:
def to_tensor(array):
    return torch.tensor(array, dtype=torch.float32)

X_train_t, y_train_t = to_tensor(X_train), to_tensor(y_train)
X_val_t, y_val_t = to_tensor(X_val), to_tensor(y_val)
X_test_t, y_test_t = to_tensor(X_test), to_tensor(y_test)

# LSTMs want (batch, seq_length, features); the MLP wants the flat window.
X_train_seq = X_train_t.unsqueeze(-1)
X_val_seq = X_val_t.unsqueeze(-1)
X_test_seq = X_test_t.unsqueeze(-1)

y_test_original = scaler.inverse_transform(y_test.reshape(-1, 1)).ravel()
print(f"MLP input {tuple(X_train_t.shape)}, LSTM input {tuple(X_train_seq.shape)}")

**Question.** The scaler's range comes from the training split, so a
test value outside it scales beyond [0, 1]. Is that a bug? What would clipping
cost you?

## 2. Multi-layer perceptron

In [ ]:
class MLPForecaster(nn.Module):
    """Flatten the lookback window and push it through dense layers."""

    def __init__(self, input_size, hidden_sizes, output_size, dropout_rate=0.2):
        super().__init__()
        layers = []
        previous = input_size
        for width in hidden_sizes:
            layers += [nn.Linear(previous, width), nn.ReLU(),
                       nn.Dropout(dropout_rate)]
            previous = width
        layers.append(nn.Linear(previous, output_size))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)


def count_parameters(model):
    """Number of trainable parameters in a module."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [ ]:
def train_model(model, X_train, y_train, X_val, y_val, epochs=100, lr=0.001,
                batch_size=32, verbose=True):
    """Train with Adam and MSE; return (train_losses, val_losses) per epoch."""
    loader = DataLoader(TensorDataset(X_train, y_train), batch_size=batch_size,
                        shuffle=True)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=10,
                                                     factor=0.5)
    train_losses, val_losses = [], []

    for epoch in range(epochs):
        model.train()
        running = 0.0
        for batch_x, batch_y in loader:
            optimizer.zero_grad()
            loss = criterion(model(batch_x), batch_y)
            loss.backward()
            optimizer.step()
            running += loss.item()

        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(X_val), y_val).item()

        train_losses.append(running / len(loader))
        val_losses.append(val_loss)
        scheduler.step(val_loss)

        if verbose and (epoch + 1) % 20 == 0:
            print(f"epoch {epoch + 1:>4}/{epochs}  "
                  f"train {train_losses[-1]:.6f}  val {val_loss:.6f}")

    return train_losses, val_losses


def plot_history(train_losses, val_losses, title):
    plt.figure(figsize=(12, 4))
    plt.plot(train_losses, label="train")
    plt.plot(val_losses, label="validation")
    plt.xlabel("epoch")
    plt.ylabel("MSE loss")
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
set_seed()
mlp = MLPForecaster(input_size=SEQ_LENGTH, hidden_sizes=[64, 32], output_size=1)
print(mlp)
print(f"\ntrainable parameters: {count_parameters(mlp):,}")

mlp_history = train_model(mlp, X_train_t, y_train_t, X_val_t, y_val_t,
                          epochs=60, lr=0.001)
plot_history(*mlp_history, "MLP")

## 3. Recurrent networks

An MLP sees the window as 30 unordered numbers. An LSTM reads them in order and
carries state between steps.

The bidirectional variant also reads the window backwards. Legitimate here: the
window holds only values from before the point being predicted. Reading it
backwards is not the same as reading the future.

In [ ]:
class LSTMForecaster(nn.Module):
    """LSTM over the lookback window; predict from the LAST time step.

    forward() takes x of shape (batch, seq_length, 1), returns
    (batch, output_size).

    Feed `output[:, -1, :]` through the linear layer. An earlier step, or an
    average over steps, changes what the model is allowed to know.
    """

    def __init__(self, input_size=1, hidden_size=64, num_layers=2,
                 output_size=1, dropout_rate=0.2):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.lstm = nn.LSTM(
            input_size=input_size, hidden_size=hidden_size,
            num_layers=num_layers, batch_first=True,
            dropout=dropout_rate if num_layers > 1 else 0.0,
        )
        # TODO: define self.fc, the linear layer mapping hidden_size -> output_size
        raise NotImplementedError

    def forward(self, x):
        # TODO
        raise NotImplementedError


class BiLSTMForecaster(nn.Module):
    """Bidirectional LSTM over the lookback window.

    A bidirectional LSTM emits 2 * hidden_size features, so size the linear
    layer for that.
    """

    def __init__(self, input_size=1, hidden_size=64, num_layers=2,
                 output_size=1, dropout_rate=0.2):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.lstm = nn.LSTM(
            input_size=input_size, hidden_size=hidden_size,
            num_layers=num_layers, batch_first=True, bidirectional=True,
            dropout=dropout_rate if num_layers > 1 else 0.0,
        )
        # TODO: self.fc — mind the factor of 2
        raise NotImplementedError

    def forward(self, x):
        # TODO
        raise NotImplementedError

In [ ]:
set_seed()
lstm = LSTMForecaster(hidden_size=32, num_layers=2)
bilstm = BiLSTMForecaster(hidden_size=32, num_layers=2)
print(f"LSTM   parameters: {count_parameters(lstm):,}")
print(f"BiLSTM parameters: {count_parameters(bilstm):,}")

lstm_history = train_model(lstm, X_train_seq, y_train_t, X_val_seq, y_val_t,
                           epochs=40, lr=0.01)
plot_history(*lstm_history, "LSTM")

In [ ]:
set_seed()
bilstm_history = train_model(bilstm, X_train_seq, y_train_t, X_val_seq, y_val_t,
                             epochs=40, lr=0.01)
plot_history(*bilstm_history, "BiLSTM")

## 4. Evaluation and comparison

### 4.1 Predictions in original units

In [ ]:
def predict(model, X, scaler):
    """Predict and undo the scaling; return a 1-D array in original units."""
    # TODO: model.eval() and torch.no_grad() first — otherwise dropout stays on
    # and the same input gives a different answer every call.
    raise NotImplementedError


def evaluate(y_true, y_pred):
    """Return {"rmse", "mae", "mape"} in original units."""
    # TODO: MAPE is mean(|actual - predicted| / |actual|) as a percentage.
    raise NotImplementedError

In [ ]:
predictions = {
    "MLP": predict(mlp, X_test_t, scaler),
    "LSTM": predict(lstm, X_test_seq, scaler),
    "BiLSTM": predict(bilstm, X_test_seq, scaler),
}

# The baseline every one of them has to beat.
naive = np.roll(y_test_original, 1)
naive[0] = y_test_original[0]
predictions["naive (y[t-1])"] = naive

scores = pd.DataFrame({name: evaluate(y_test_original, values)
                       for name, values in predictions.items()}).T
print(scores.sort_values("rmse").round(4).to_string())

### 4.2 Visual comparison

In [ ]:
plt.figure(figsize=(20, 6))
window = slice(0, 200)
plt.plot(y_test_original[window], color="black", linewidth=1.5, label="actual")
for name in ("MLP", "LSTM", "BiLSTM"):
    plt.plot(predictions[name][window], linewidth=1, alpha=0.8, label=name)
plt.title("Test set — first 200 days")
plt.legend();

### 4.3 Model performance comparison

In [ ]:
ordered = scores.sort_values("rmse")

fig, axes = plt.subplots(1, 3, figsize=(20, 4))
for ax, metric in zip(axes, ("rmse", "mae", "mape")):
    ax.barh(ordered.index[::-1], ordered[metric][::-1])
    ax.set_title(metric.upper())
plt.tight_layout();

### 4.4 Residual analysis

In [ ]:
best_name = ordered.index[0]
residuals = y_test_original - predictions[best_name]

fig, axes = plt.subplots(1, 3, figsize=(20, 4))
axes[0].plot(residuals); axes[0].axhline(0, color="red", linestyle="--")
axes[0].set_title(f"Residuals over time — {best_name}")
axes[1].hist(residuals, bins=40); axes[1].set_title("Residual distribution")
axes[2].scatter(y_test_original, predictions[best_name], s=5, alpha=0.4)
limits = [y_test_original.min(), y_test_original.max()]
axes[2].plot(limits, limits, "r--")
axes[2].set_xlabel("actual"); axes[2].set_ylabel("predicted")
axes[2].set_title("Predicted vs actual")
plt.tight_layout()

print(f"best model: {best_name}")
print(f"residual mean {residuals.mean():.4f}, std {residuals.std():.4f}")

### 4.5 Questions for analysis

1. **Did any network beat the naive `y[t-1]` baseline?** Give the numbers. If
   not, that is a real finding, not a failed practice — what does it say about
   day-to-day temperature?
2. The LSTM knows the window is ordered; the MLP does not. Did that help? What
   property of this series explains the result?
3. BiLSTM carries more than twice the parameters of LSTM. Did it earn them?
4. Read the loss curves. Does validation loss flatten, keep falling, or turn
   back up — and what would you change in each case?
5. Rank your best network against Holt-Winters (Practice 2), ARIMA (Practice 4)
   and Ridge (Practice 5) by RMSE. Now rank them by build time. Same order?
6. Refit the scaler on the whole series, rerun, and report how far the scores
   move. Big enough to notice? Does that make the shortcut acceptable?

In [ ]:
# your code here — free exploration, not graded